# ReceiptVLM — re-train the LoRA adapter for CUDA (QLoRA / NF4)Re-fits the receipt-extraction LoRA with `transformers` + `peft` so it can be served offApple Silicon. Hyperparameters mirror `src/train.py`'s production run exactly; the onlydeliberate changes are the framework and the base checkpoint.## Why this existsThe original adapter was trained by QLoRA against `mlx-community/Qwen2.5-VL-3B-Instruct-4bit`.Converting it to peft format is algebraically exact (verified to 1.5e-8), but replaying iton an **fp16** base scores **0.288 micro-F1 against the MLX run's 0.785** — the adapter wasfit to compensate one specific set of quantized weights, and a ~1% perturbation aimed atthose weights does nothing useful to different ones. The failure is visible in the output:the model still reads the receipt but reverts to the base model's conventions(`CHO EUN KOREAN RESTAUR` where the WildReceipt target is `CHOEUN KOREANRESTAURAN`).**So the rule this notebook exists to enforce: train against the base you will serve.**It trains in 4-bit NF4 and you serve in 4-bit NF4. NF4 is not needed for memory on a48 GB ZeroGPU slice — it is here to guarantee the train/serve bases match.## Runtime~1140 training receipts x 2 epochs = 2280 steps at batch size 1. Expect **2-4 hours** on aKaggle P100 or T4, well inside the 9-12h session cap and ~10% of the weekly GPU quota.Use **Save & Run All (Commit)** so the run survives closing the tab.## Inputs this notebook needs**WildReceipt images** are downloaded below (~179 MB), or attach them as a Kaggle Datasetto skip the download on re-runs.Everything else goes in one small Kaggle Dataset (~1.5 MB total). Only `train.jsonl` isrequired to train; the rest let the final cell score the result in-session, which mattersbecause the accuracy gate cannot run on Apple Silicon (bitsandbytes has no MPS backend, soan NF4-trained adapter can only be scored on a GPU host — here).| file | from | needed for ||---|---|---|| `train.jsonl` | `data/processed/` | training (required) || `test.jsonl` | `data/processed/` | scoring || `finetuned_test.jsonl` | `data/processed/` | the paired comparison against the MLX run || `eval.py` | `src/` | the scoring harness itself || `repair.py`, `schema.py`, `zeroshot.py` | `src/` | `eval.py`'s siblings, imported by bare name |Using the project's own `eval.py` rather than a re-implementation is deliberate: are-derived scorer would produce a number that is not comparable to the 0.781 in`RESULTS.md`, which defeats the point of measuring.

### Pick the T4, not the P100Set **Accelerator → GPU T4 x2** in the sidebar. Kaggle's P100 is Pascal (`sm_60`), andcurrent PyTorch wheels and bitsandbytes builds no longer ship kernels for it — on a P100the model load dies with `CUDA error: no kernel image is available for execution on thedevice` after downloading 7 GB. The T4 is Turing (`sm_75`) and is supported. Only one ofthe two T4s is used.The cell below checks this before anything expensive happens.

In [ ]:
# Kaggle's image already has a torch matched to its drivers, so torch is *pinned to the# installed version* via a constraints file. Without that, resolving the packages below# can pull a newer torch wheel built for a narrower set of GPU architectures, which then# fails at model-load time with "no kernel image is available".import torchopen("constraints.txt", "w").write(f"torch=={torch.__version__.split('+')[0]}\n")

In [ ]:
!pip install -q -U -c constraints.txt "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes>=0.44" "safetensors>=0.4"

In [ ]:
# torch is deliberately NOT reloaded here. Re-executing its module body re-registers the# `triton` TORCH_LIBRARY namespace, which the C++ dispatcher rejects:#   "Only a single TORCH_LIBRARY can be used to register the namespace triton"# The constraints file above means torch should not have changed at all; if it somehow# did, the fix is a session restart, not a reload.import torchimport transformers, peft, bitsandbytespinned = open("constraints.txt").read().strip().split("==")[1]if torch.__version__.split("+")[0] != pinned:    raise SystemExit(        f"torch changed from {pinned} to {torch.__version__} despite the constraint. "        "Restart the session (Run -> Restart session) and re-run from the top -- the "        "already-imported torch cannot be swapped in place."    )print("torch", torch.__version__, "| cuda", torch.version.cuda)print("transformers", transformers.__version__, "| peft", peft.__version__,      "| bnb", bitsandbytes.__version__)if not torch.cuda.is_available():    raise SystemExit("No GPU. Set Accelerator -> GPU T4 x2 in the sidebar.")name = torch.cuda.get_device_name(0)major, minor = torch.cuda.get_device_capability(0)sm = f"sm_{major}{minor}"arches = torch.cuda.get_arch_list()print(f"device {name}  capability {sm}")print("torch arch list:", arches)# Fail here rather than 7 GB into the model download.if sm not in arches:    raise SystemExit(        f"This torch build has no kernels for {name} ({sm}); it targets {arches}.\n"        "Switch Accelerator to 'GPU T4 x2' (sm_75) in the sidebar and re-run. Kaggle's "        "P100 is sm_60, which current torch and bitsandbytes wheels have dropped."    )if (major, minor) < (7, 5):    raise SystemExit(        f"{name} is {sm}; bitsandbytes 4-bit needs sm_75 or newer. Switch to "        "'GPU T4 x2'."    )# T4 is pre-Ampere: no bf16, hence fp16 everywhere below, and no flash-attention 2.print("bf16 supported:", torch.cuda.is_bf16_supported())print("\nenvironment OK")

## Configuration — mirrors `src/train.py`'s production run

In [ ]:
import json, math, random, re, shutil, timefrom pathlib import Path# --- the exact prompt and key order the model was fine-tuned on. Key order matters: the# training target is serialized in this order, so the model learned to emit it this way.SCHEMA_KEYS = ["store", "date", "tax", "tip", "subtotal", "total", "line_items"]PROMPT = ("Extract the receipt fields as JSON with keys store, date, tax, tip, "          "subtotal, total, line_items (each {name, price}). Use null for missing "          "scalar fields and [] for no line items.")# --- hyperparameters, matching src/train.py + checkpoints/final/adapter_config.jsonBASE_MODEL    = "Qwen/Qwen2.5-VL-3B-Instruct"LORA_RANK     = 4        # adapter_config.json: rank 4LORA_ALPHA    = 0.5      # adapter_config.json: alpha 0.5  -> scale alpha/rank = 0.125LORA_DROPOUT  = 0.05LR            = 1e-4     # train.py --lr defaultEPOCHS        = 2        # 1140 train records x 2 = 2280 steps, matching training_log.jsonVAL_FRAC      = 0.1SEED          = 0IMAGE_RESIZE  = (768, 1024)   # train.py --image-resize defaultMAX_GRAD_VALUE = 1.0     # train.py passes clip_gradients=1.0EVAL_EVERY    = 100      # train.py uses 25; raised here because each eval costs GPU timeEVAL_BATCHES  = 24SAVE_EVERY    = 250      # train.py uses 50PRINT_EVERY   = 25KEEP_LAST     = 2OUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("out")CKPT_ROOT = OUT_ROOT / "checkpoints"CKPT_ROOT.mkdir(parents=True, exist_ok=True)print("scale alpha/rank =", LORA_ALPHA / LORA_RANK, "(must equal the MLX run's 0.125)")print("writing to", OUT_ROOT)

## Data

In [ ]:
INPUT_ROOT = Path("/kaggle/input")def find_input(name, must_contain=None):    """Locate a file or directory anywhere under /kaggle/input, then the local repo.    Searched at any depth on purpose: Kaggle preserves folder structure on upload, so the    same files land at <dataset>/train.jsonl if you upload them flat but at    <dataset>/data/processed/train.jsonl if you upload the repo folders -- and a    fixed-depth glob silently misses the second case.    """    def ok(p):        return (p / must_contain).exists() if must_contain else p.exists()    if INPUT_ROOT.exists():        for p in sorted(INPUT_ROOT.rglob(name)):            if ok(p):                return p    for p in (Path("data/processed") / name, Path("data") / name,              Path("src") / name, Path(name)):        if p.exists() and ok(p):            return p    return Nonedef attached_inputs():    return sorted(str(p.relative_to(INPUT_ROOT)) for p in INPUT_ROOT.rglob("*")                  if p.is_file()) if INPUT_ROOT.exists() else []# --- locate train.jsonl (uploaded as a Kaggle Dataset) -----------------------------train_jsonl = find_input("train.jsonl")if train_jsonl is None:    listing = attached_inputs()    raise SystemExit(        "Could not find train.jsonl. Upload data/processed/train.jsonl as a Kaggle "        "Dataset and attach it via '+ Add Input', or regenerate it with src/prep.py.\n"        f"Files currently attached ({len(listing)}): {listing[:25]}"    )print("train.jsonl:", train_jsonl)# --- locate or download the WildReceipt images -------------------------------------img_root = find_input("wildreceipt", must_contain="image_files")if img_root is None and (OUT_ROOT / "wildreceipt" / "image_files").exists():    img_root = OUT_ROOT / "wildreceipt"if img_root is None:    # Python rather than `!curl`/`!tar`: no shell quoting of paths, and it behaves the    # same in Kaggle, Colab and plain Jupyter.    import tarfile, urllib.request    dest = OUT_ROOT / "wildreceipt"    dest.mkdir(parents=True, exist_ok=True)    tar_path = OUT_ROOT / "wildreceipt.tar"    if not tar_path.exists():        print("downloading WildReceipt (~179 MB)...")        urllib.request.urlretrieve(            "https://download.openmmlab.com/mmocr/data/wildreceipt.tar", tar_path)    print("extracting...")    with tarfile.open(tar_path) as tf:        # --strip-components=1 equivalent: the archive nests everything under one dir.        for member in tf.getmembers():            parts = Path(member.name).parts            if len(parts) <= 1:                continue            member.name = str(Path(*parts[1:]))            tf.extract(member, dest)    img_root = destprint("images:", img_root, "| image_files present:", (img_root / "image_files").exists())

In [ ]:
# --- split, reproducing src/train.py's load_split() byte for byte ------------------# Same shuffle seed and the same "first val_frac becomes validation" rule, so this run# trains on exactly the receipts the MLX run trained on and validates on the same held-out# set. Any other split would make the two runs incomparable.records = [json.loads(l) for l in train_jsonl.open() if l.strip()]rng = random.Random(SEED)rng.shuffle(records)n_val = max(1, round(len(records) * VAL_FRAC)) if len(records) > 1 else 0val_records, train_records = records[:n_val], records[n_val:]missing = [r for r in records if not (img_root / r["image_id"]).exists()]print(f"total={len(records)}  train={len(train_records)}  val={len(val_records)}")print(f"missing images: {len(missing)}")print(f"total steps = {len(train_records) * EPOCHS}  (the MLX run logged 2280)")assert not missing, f"{len(missing)} images missing, e.g. {missing[0]['image_id']}"def target_json(record: dict) -> str:    """The training target: schema fields only, fixed key order, no ASCII escaping."""    return json.dumps({k: record[k] for k in SCHEMA_KEYS}, ensure_ascii=False)print("\nexample target:\n", target_json(train_records[0])[:220], "...")

## DatasetTwo details here are load-bearing:**Image sizing** is a port of `mlx_vlm.utils.resize_image` — it scales to *fit inside*768x1024 with aspect preserved, it does **not** stretch to that shape, and it does notclamp the ratio to <= 1 (a receipt smaller than the box gets upscaled, which is what theoriginal run trained on).**Completion-only loss.** `src/train.py` used mlx_vlm's `train_on_completions=True`, whichmasks everything up to the assistant token. Here the prompt is processed on its own to getits exact token length — including the image placeholder expansion, which depends on theimage's patch grid — and those positions are set to -100. Same effect, and it cannotmis-fire on a receipt whose text happens to contain the word "assistant".

In [ ]:
from PIL import Imagefrom torch.utils.data import Datasetdef fit_within(img, max_w, max_h):    """Port of mlx_vlm.utils.resize_image: fit the box, aspect preserved, no clamp."""    ratio = min(max_w / img.width, max_h / img.height)    return img.resize((int(img.width * ratio), int(img.height * ratio)))class ReceiptDataset(Dataset):    def __init__(self, records, processor, img_root, image_resize):        self.records = records        self.processor = processor        self.img_root = Path(img_root)        self.image_resize = image_resize        self.tokenizer = processor.tokenizer        # Learn the stop token so the model is trained to terminate, not to run to the        # token cap and get truncated mid-JSON.        self.im_end = self.tokenizer.convert_tokens_to_ids("<|im_end|>")    def __len__(self):        return len(self.records)    def __getitem__(self, idx):        rec = self.records[idx]        img = Image.open(self.img_root / rec["image_id"]).convert("RGB")        img = fit_within(img, *self.image_resize)        messages = [{"role": "user", "content": [{"type": "image"},                                                 {"type": "text", "text": PROMPT}]}]        prompt_text = self.processor.apply_chat_template(            messages, tokenize=False, add_generation_prompt=True)        enc = self.processor(text=[prompt_text], images=[img], return_tensors="pt")        prompt_ids = enc["input_ids"][0]        target_ids = torch.tensor(            self.tokenizer(target_json(rec), add_special_tokens=False)["input_ids"]            + [self.im_end], dtype=prompt_ids.dtype)        input_ids = torch.cat([prompt_ids, target_ids])        labels = torch.cat([            torch.full((len(prompt_ids),), -100, dtype=torch.long),            target_ids.to(torch.long),        ])        return {            "input_ids": input_ids.unsqueeze(0),            "attention_mask": torch.ones_like(input_ids).unsqueeze(0),            "labels": labels.unsqueeze(0),            "pixel_values": enc["pixel_values"],            "image_grid_thw": enc["image_grid_thw"],        }

In [ ]:
from transformers import AutoProcessorprocessor = AutoProcessor.from_pretrained(BASE_MODEL)train_ds = ReceiptDataset(train_records, processor, img_root, IMAGE_RESIZE)val_ds = ReceiptDataset(val_records, processor, img_root, IMAGE_RESIZE)# Sanity-check the masking before spending hours on it: the unmasked span must be exactly# the JSON completion plus its stop token, and nothing from the prompt.s = train_ds[0]n_sup = int((s["labels"] != -100).sum())decoded = processor.tokenizer.decode(s["labels"][0][s["labels"][0] != -100])print("sequence length:", s["input_ids"].shape[1])print("supervised tokens:", n_sup, f"({100 * n_sup / s['input_ids'].shape[1]:.1f}% of the sequence)")print("vision patches:", tuple(s["image_grid_thw"][0].tolist()))print("\nsupervised text (should be the JSON target + stop token):")print(decoded[:300])assert decoded.startswith("{\"store\"") and decoded.rstrip().endswith("<|im_end|>"), \    "masking is off -- the supervised span is not exactly the completion"print("\nmasking OK")

## Model`target_modules` is an **anchored regex**, not a list of suffixes. Qwen2.5-VL's visiontower has 161 linear layers whose names end in `gate_proj` / `up_proj` / `down_proj` /`qkv` / `proj`, so a bare suffix list would inject adapters into the vision encoder too —which the original run did not do (it targeted `model.language_model` only, 252 modules).Anchoring keeps this run comparable to that one and keeps the adapter loadable by`src/backend_hf.py`.

In [ ]:
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGenerationfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training# NF4, and the serving side must load NF4 too. This is the whole point of the notebook.bnb = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_quant_type="nf4",    bnb_4bit_use_double_quant=True,    # fp16 rather than bf16: T4/P100 are pre-Ampere and have no bf16 support.    bnb_4bit_compute_dtype=torch.float16,)model = Qwen2_5_VLForConditionalGeneration.from_pretrained(    BASE_MODEL, quantization_config=bnb, dtype=torch.float16,    attn_implementation="sdpa",   # flash-attention 2 needs Ampere+)model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)model.config.use_cache = Falsemodel.enable_input_require_grads()   # required for checkpointing through a frozen baseTARGET_REGEX = (r"model\.language_model\.layers\.\d+\."                r"(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)")model = get_peft_model(model, LoraConfig(    r=LORA_RANK,    lora_alpha=LORA_ALPHA,     # peft scales by lora_alpha / r, as MLX did by alpha / rank    lora_dropout=LORA_DROPOUT,    target_modules=TARGET_REGEX,    bias="none",    task_type="CAUSAL_LM",))n_lora = sum(1 for _, m in model.named_modules()             if hasattr(getattr(m, "lora_A", None), "keys"))n_vision = sum(1 for n, m in model.named_modules()               if hasattr(getattr(m, "lora_A", None), "keys") and ".visual." in n)print(f"LoRA modules injected: {n_lora}   in the vision tower: {n_vision}")model.print_trainable_parameters()assert n_lora == 252, f"expected 252 adapted modules (the MLX run's count), got {n_lora}"assert n_vision == 0, "the regex leaked into the vision tower"

## Training loopDeliberately a plain loop rather than `Trainer`, to keep `src/train.py`'s three unusualchoices intact:- **`Adam`, not `AdamW`** — no weight decay, constant lr, no warmup or schedule.- **Element-wise gradient clipping at 1.0.** mlx's `clip_gradients=1.0` maps to  `tree_map(mx.clip(g, -1.0, 1.0))`, which clips each element — that is  `clip_grad_value_`, *not* the global-norm `clip_grad_norm_` most recipes use.- **NaN/Inf steps are skipped, not clamped.** If the loss or any gradient is non-finite  the optimizer does not step and the record is logged, matching `guarded_train_step`.

In [ ]:
device = "cuda"optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)scaler = torch.amp.GradScaler("cuda")trainable = [p for p in model.parameters() if p.requires_grad]def to_device(batch):    return {k: v.to(device) for k, v in batch.items()}@torch.no_grad()def val_loss(max_batches=EVAL_BATCHES):    """Mean loss over the held-out split, ignoring non-finite batches (as train.py does)."""    model.eval()    n = min(max_batches, len(val_ds))    losses = []    for i in range(n):        with torch.autocast("cuda", dtype=torch.float16):            loss = model(**to_device(val_ds[i])).loss        v = loss.item()        if math.isfinite(v):            losses.append(v)    model.train()    return sum(losses) / len(losses) if losses else float("nan")def save_checkpoint(step):    ckpt = CKPT_ROOT / f"step_{step}"    model.save_pretrained(ckpt)          # peft format: adapter_model.safetensors + config    kept = sorted(CKPT_ROOT.glob("step_*"), key=lambda p: int(p.name.split("_")[1]))    for old in kept[:-KEEP_LAST]:        shutil.rmtree(old, ignore_errors=True)    return ckpt

In [ ]:
total_steps = len(train_ds) * EPOCHSorder = list(range(len(train_ds)))rng = random.Random(SEED)history, skipped = [], 0model.train()t_start = time.time()print(f"training {total_steps} steps (batch size 1)\n")for step in range(total_steps):    pos = step % len(order)    if pos == 0:        rng.shuffle(order)          # reshuffle each epoch, same as train.py    batch = to_device(train_ds[order[pos]])    with torch.autocast("cuda", dtype=torch.float16):        loss = model(**batch).loss    optimizer.zero_grad(set_to_none=True)    scaler.scale(loss).backward()    scaler.unscale_(optimizer)      # unscale before inspecting/clipping raw gradients    loss_val = loss.item()    finite = math.isfinite(loss_val) and all(        torch.isfinite(p.grad).all().item() for p in trainable if p.grad is not None)    if finite:        # element-wise clip, mirroring mlx's tree_map(mx.clip(g, -1, 1))        torch.nn.utils.clip_grad_value_(trainable, MAX_GRAD_VALUE)        scaler.step(optimizer)    else:        skipped += 1        print(f"step {step}/{total_steps} SKIPPED (non-finite loss/grad, "              f"record={train_records[order[pos]]['image_id']})")    scaler.update()    history.append({"step": step, "loss": loss_val, "applied": bool(finite)})    if step % PRINT_EVERY == 0:        el = time.time() - t_start        rate = (step + 1) / el        eta = (total_steps - step - 1) / rate / 60        print(f"step {step:>5}/{total_steps}  loss {loss_val:.4f}  "              f"{rate:.2f} it/s  eta {eta:.0f}m")    if step and step % EVAL_EVERY == 0:        vl = val_loss()        history[-1]["val_loss"] = vl        print(f"  step {step}: val_loss {vl:.4f}")    if step and step % SAVE_EVERY == 0:        print("  checkpoint ->", save_checkpoint(step))print(f"\ndone in {(time.time() - t_start) / 60:.0f}m  ({skipped} steps skipped)")print("final val_loss:", val_loss())

## Save the adapter

In [ ]:
final_dir = OUT_ROOT / "final_peft"model.save_pretrained(final_dir)(OUT_ROOT / "training_log.json").write_text(json.dumps(history))cfg = json.loads((final_dir / "adapter_config.json").read_text())print("saved to", final_dir)for f in sorted(final_dir.iterdir()):    print(f"  {f.name}  {f.stat().st_size / 1e6:.1f} MB")print("\nr / lora_alpha:", cfg["r"], "/", cfg["lora_alpha"],      "-> scale", cfg["lora_alpha"] / cfg["r"])print("target_modules:", cfg["target_modules"])# The scale must match the original run, or the adapter is a different model.assert cfg["lora_alpha"] / cfg["r"] == 0.125, "scale drifted from the MLX run's 0.125"

## Inference helperOne generation path, shared by the qualitative check and the scoring gate below, so thetwo cannot diverge. Decoding matches `src/zeroshot.py`: greedy, 1536 new tokens, **norepetition penalty** — that file records that a penalty collapsed micro-F1 from 0.525 to0.069, because JSON is inherently repetitive and penalising repeated tokens fights theschema rather than the content.

In [ ]:
model.eval()model.config.use_cache = Truedef generate_raw(image_path, max_new_tokens=1536):    """Return the model's raw completion for one receipt image."""    img = fit_within(Image.open(image_path).convert("RGB"), *IMAGE_RESIZE)    messages = [{"role": "user", "content": [{"type": "image"},                                             {"type": "text", "text": PROMPT}]}]    text = processor.apply_chat_template(messages, tokenize=False,                                         add_generation_prompt=True)    enc = processor(text=[text], images=[img], return_tensors="pt").to(device)    with torch.inference_mode():        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)    return processor.batch_decode(out[:, enc["input_ids"].shape[1]:],                                   skip_special_tokens=True)[0]print("ready")

## Qualitative checkWhat to look for is not exact matches but whether the **learned conventions came back** —spaces stripped from store and item names, and the extended price rather than the unitprice. That is precisely what the fp16-transferred adapter lost, so it is the fastestsignal that this retrain worked.

In [ ]:
for rec in val_records[:4]:    pred = generate_raw(img_root / rec["image_id"])    gold = {k: rec[k] for k in SCHEMA_KEYS}    print("=" * 78)    print("GOLD:", json.dumps(gold, ensure_ascii=False)[:300])    print("PRED:", pred[:300])    print()

## The accuracy gateScores the test split with the project's own `eval.py` — per-field micro-F1 with apercentile bootstrap CI, plus a paired bootstrap against the MLX run on the same receipts.It reads the **first `SCORE_LIMIT` receipts of `test.jsonl` in file order**, which is what`scripts/validate_peft_adapter.py` does, so the number here is directly comparable to the0.288 the fp16-transferred adapter scored and to the MLX run's 0.785 on those same 30.If the scoring files were not attached this cell prints what to add and skips, rather thanfailing and making a good training run look broken.

In [ ]:
import sysSCORE_LIMIT = 30   # same receipts scripts/validate_peft_adapter.py useseval_py = find_input("eval.py")test_jsonl = find_input("test.jsonl")reference = find_input("finetuned_test.jsonl")# eval.py's siblings import each other by bare name (run-as-script style), so they have to# sit in the same directory as eval.py -- that directory is what goes on sys.path. They# may live anywhere in the dataset, together.SIBLINGS = ["repair.py", "schema.py", "zeroshot.py"]missing_siblings = ([s for s in SIBLINGS if not (eval_py.parent / s).exists()]                    if eval_py else SIBLINGS)if eval_py is None or test_jsonl is None or missing_siblings:    print("SKIPPING the gate -- scoring inputs not attached.\n")    print("Add these to the Kaggle Dataset, then re-run this cell:")    print("  data/processed/test.jsonl              (ground truth)")    print("  data/processed/finetuned_test.jsonl    (MLX reference, for the paired test)")    print("  src/eval.py src/repair.py src/schema.py src/zeroshot.py")    print("\nThe four .py files must end up in the same folder as each other; the")    print(".jsonl files can sit anywhere in the dataset.")    print(f"\nfound: eval.py={eval_py}  test.jsonl={test_jsonl}  reference={reference}")    if eval_py is not None and missing_siblings:        print(f"missing next to eval.py: {missing_siblings}")else:    sys.path.insert(0, str(eval_py.parent))    from eval import bootstrap_micro_f1, evaluate, paired_bootstrap_test    from repair import repair_json    from zeroshot import normalize            # mlx import is guarded, so this is safe here    gold_all = {json.loads(l)["image_id"]: json.loads(l)                for l in test_jsonl.open() if l.strip()}    score_ids = list(gold_all)[:SCORE_LIMIT]    absent = [i for i in score_ids if not (img_root / i).exists()]    assert not absent, f"{len(absent)} test images missing, e.g. {absent[0]}"    preds, statuses = {}, {}    t0 = time.time()    for n, iid in enumerate(score_ids, 1):        raw = generate_raw(img_root / iid)        parsed, status = repair_json(raw)        statuses[status] = statuses.get(status, 0) + 1        preds[iid] = {"image_id": iid, **normalize(parsed)}        print(f"  [{n}/{len(score_ids)}] {status:<24} {iid.split('/')[-1][:26]}", flush=True)    print(f"\nscored {len(preds)} receipts in {time.time() - t0:.0f}s | repair: {statuses}")    gold = {i: gold_all[i] for i in preds}    per_field, micro, n_scored = evaluate(gold, preds)    lo, hi = bootstrap_micro_f1(gold, preds, n=1000)    print(f"\n{'NF4 retrain':<16} micro-F1 {micro[2]:.3f}  "          f"95% CI [{lo:.3f}, {hi:.3f}]  (n={n_scored})")    for field, (p, r, f1) in sorted(per_field.items()):        print(f"  {field:<18} P {p:.3f}  R {r:.3f}  F1 {f1:.3f}")    if reference is not None:        ref_all = {json.loads(l)["image_id"]: json.loads(l)                   for l in reference.open() if l.strip()}        shared = [i for i in preds if i in ref_all]        if shared:            g = {i: gold_all[i] for i in shared}            a = {i: ref_all[i] for i in shared}       # MLX run            b = {i: preds[i] for i in shared}         # this retrain            _, ref_micro, _ = evaluate(g, a)            _, own_micro, _ = evaluate(g, b)            res = paired_bootstrap_test(g, a, b, n=1000)   # reports b - a            print(f"\nPaired against {reference.name} on {len(shared)} receipts:")            print(f"  MLX reference  micro-F1 {ref_micro[2]:.3f}")            print(f"  NF4 retrain    micro-F1 {own_micro[2]:.3f}")            print(f"  delta {res['mean_diff']:+.3f}  "                  f"95% CI [{res['ci'][0]:+.3f}, {res['ci'][1]:+.3f}]  "                  f"p={res['p_approx']:.3f}")            regressed = res["mean_diff"] < 0 and res["p_approx"] < 0.05            print("\n" + ("GATE FAIL: still significantly behind the MLX run."                          if regressed else                          "GATE PASS: no significant regression vs. the MLX run."))            print("For reference, the fp16-transferred adapter scored 0.288 here.")    else:        print("\nNo MLX reference attached; skipped the paired comparison.")    # Keep the predictions so the run can be re-scored later without another GPU session.    (OUT_ROOT / "nf4_test_predictions.jsonl").write_text(        "".join(json.dumps(v) + "\n" for v in preds.values()))    print("\nwrote", OUT_ROOT / "nf4_test_predictions.jsonl")model.config.use_cache = False

## Next stepsIf the gate passed:1. Download `final_peft/` from the notebook output (**Output** tab, or the   `/kaggle/working` file browser). Also grab `training_log.json` and   `nf4_test_predictions.jsonl`.2. Drop it into the repo as `checkpoints/final_peft`, replacing the converted adapter.3. Serve it **in NF4**, matching what it was trained on:   ```bash   RECEIPTVLM_LOAD_4BIT=1 python app.py   ```   On the Space, set `RECEIPTVLM_LOAD_4BIT=1` as a variable and remove   `RECEIPTVLM_SAMPLES_ONLY`, then rebuild:   ```bash   python scripts/build_space.py --out build/space --force   ```   Do not serve it in fp16. That is the exact mismatch this notebook exists to avoid, and   it is why `scripts/validate_peft_adapter.py --load-4bit` refuses to run without CUDA.If the gate failed, check the loss curve in `training_log.json` — but **do not compare itdirectly to the MLX run's numbers.** The two losses use different denominators:- mlx_vlm computes `ce.sum() / ntoks`, where the numerator is masked to completion tokens  but `ntoks` comes from the attention mask, i.e. the whole sequence.- HuggingFace's `model(labels=...)` averages over non-`-100` positions only.Completions are roughly 10–25% of a sequence here, so the MLX figures are scaled down byabout that factor. Its logged 0.067 → 0.035 corresponds to something nearer **0.3–0.5falling to 0.15–0.25** on this notebook's scale, and the exact multiplier drifts perreceipt because the supervised fraction does.So judge the **trend**, not the absolute value: a loss that falls steadily and roughlyhalves is the signal. A loss that is flat or rising points at the lr or the masking(re-check the supervised-span assertion above). A healthy curve that still scores badlypoints instead at the decode path or the image sizing.